# STAC AI Workshop · NYC · 21 October 2026
## 01 · Build and inspect three LLM workflows
**Matthew Dixon, PhD, FRM · BeliefLens**

### Learning objectives
1. Implement fixed retrieval, fixed retrieval plus calculation, and an agent tool loop.
2. Inspect the exact messages, tool arguments and results exchanged with a model.
3. Distinguish correct arithmetic from justified inputs.
4. Change the model client without changing the workflow controller.

This notebook runs on Python's standard library, without keys or companion files. The default `SyntheticLLM` is a transparent, deterministic test double—not a trained language model. It reads the supplied messages and requests tools; the application actually executes those tools. Its deliberately scripted choices demonstrate control flow, not measured model superiority. Optional live mode calls OpenAI or Together directly from your Python kernel.

**Task:** Does Harbor Metals have comfortable covenant headroom? Debt is $75m, cash $15m, EBITDA $20m, and the ceiling is 3.5×. A covenant annex may exclude $5m restricted cash.

In [1]:
import json, math, re, copy, uuid
from pprint import pprint

QUESTION = "Does Harbor Metals have comfortable covenant headroom? Establish current covenant leverage, cite sources, and identify what remains unresolved."
SYSTEM = """You assess the fictional Harbor Metals case. Use only supplied documents and tool results.
For an agent run, retrieve the initial packet first. Follow any annex reference before choosing eligible cash.
Request calculate_leverage for arithmetic. If evidence is unavailable, state the gap; do not invent an amount.
A current ratio alone does not establish comfortable headroom or authorization to act.
Give a concise answer with document IDs. Tools cannot trade or release a brief."""

def make_store(restricted=True, annex_available=True, debt=75, cash=15, ebitda=20):
    docs = {
        "FILING": {"id": "FILING", "text": f"Harbor Metals: debt {debt} million; cash {cash} million; EBITDA {ebitda} million."},
        "COVENANT": {"id": "COVENANT", "text": "Ceiling 3.5 times. " + (
            "Exclude restricted cash specified in Annex A." if restricted else "All reported cash is eligible.")}}
    if restricted and annex_available:
        docs["ANNEX_A"] = {"id": "ANNEX_A", "text": "Restricted cash: 5 million."}
    return docs

def retrieve(store, document_id):
    # Deliberately small document-ID retriever. A production retriever can replace it.
    if document_id == "INITIAL":
        return {"documents": [copy.deepcopy(store[k]) for k in ("FILING", "COVENANT")]}
    return {"documents": [copy.deepcopy(store[document_id])]} if document_id in store else {"documents": [], "error": "Document unavailable", "requested": document_id}

def calculate_leverage(debt, eligible_cash, ebitda):
    values = [debt, eligible_cash, ebitda]
    if any(isinstance(x, bool) or not isinstance(x, (int, float)) or not math.isfinite(x) for x in values):
        raise ValueError("Finite numeric inputs required")
    if debt < 0 or eligible_cash < 0 or ebitda <= 0:
        raise ValueError("Debt/cash must be nonnegative and EBITDA positive")
    return {"debt": debt, "eligible_cash": eligible_cash, "ebitda": ebitda,
            "formula": "(debt - eligible_cash) / ebitda", "leverage": (debt-eligible_cash)/ebitda}

TOOLS = [
    {"type": "function", "function": {"name": "retrieve", "description": "Retrieve INITIAL for filing and covenant, or ANNEX_A for the referenced annex.",
        "parameters": {"type": "object", "properties": {"document_id": {"type": "string"}}, "required": ["document_id"], "additionalProperties": False}}},
    {"type": "function", "function": {"name": "calculate_leverage", "description": "Calculate leverage using evidence-supported eligible cash, in millions.",
        "parameters": {"type": "object", "properties": {k: {"type": "number"} for k in ["debt", "eligible_cash", "ebitda"]}, "required": ["debt", "eligible_cash", "ebitda"], "additionalProperties": False}}}
]
pprint(retrieve(make_store(), "INITIAL"))

{'documents': [{'id': 'FILING',
                'text': 'Harbor Metals: debt 75 million; cash 15 million; '
                        'EBITDA 20 million.'},
               {'id': 'COVENANT',
                'text': 'Ceiling 3.5 times. Exclude restricted cash specified '
                        'in Annex A.'}]}


### 1 · A model object with a real message/tool interface
`complete(messages, tools)` returns an assistant message: either text or tool-call requests. The synthetic model sees only messages, not the document store. Its policy below is intentionally visible. It parses this controlled document format; this is not general document understanding.

Set `skip_annex=True` to simulate a model that uses total cash without investigating the definition. The workflow controller must not silently repair the model's arguments.

In [2]:
def number(text, field):
    return float(re.search(re.escape(field) + r"[: ]+([0-9.]+)", text, re.I).group(1))

def received_payloads(messages):
    payloads=[]
    for m in messages:
        if m["role"] in ("tool", "user"):
            try:
                value=json.loads(m.get("content") or "")
                if isinstance(value, dict): payloads.append(value)
            except (ValueError, TypeError): pass
    return payloads

class SyntheticLLM:
    def __init__(self, skip_annex=False): self.skip_annex=skip_annex
    def complete(self, messages, tools=None):
        payloads=received_payloads(messages)
        docs={d["id"]:d for p in payloads for d in p.get("documents", [])}
        calculations=[p for p in payloads if "leverage" in p]
        def call(name, args):
            return {"role":"assistant", "content":None, "tool_calls":[{
                "id":"call_"+uuid.uuid4().hex[:10], "type":"function",
                "function":{"name":name,"arguments":json.dumps(args)}}]}
        if tools and "FILING" not in docs:
            return call("retrieve", {"document_id":"INITIAL"})
        needs_annex="Annex A" in docs.get("COVENANT",{}).get("text","")
        tried_annex=any(p.get("requested")=="ANNEX_A" for p in payloads)
        if tools and needs_annex and "ANNEX_A" not in docs and not self.skip_annex:
            if tried_annex:
                return {"role":"assistant","content":"HOLD calculation claim: Annex A is unavailable; eligible cash is unresolved [COVENANT]."}
            return call("retrieve", {"document_id":"ANNEX_A"})
        if tools and not calculations:
            f=docs["FILING"]["text"]
            cash=number(f,"cash") - (number(docs["ANNEX_A"]["text"],"Restricted cash") if "ANNEX_A" in docs else 0)
            return call("calculate_leverage", {"debt":number(f,"debt"),"eligible_cash":cash,"ebitda":number(f,"EBITDA")})
        if calculations:
            result=calculations[-1]
            text=f"Calculator reports {result['leverage']:.2f}× using cash {result['eligible_cash']}m. Sources: {', '.join(docs)}. Comfort and release still need scenario review and approval."
        else:
            text="Retrieved the filing and covenant, but this workflow supplies no executed calculation. Current headroom needs verification; any referenced annex must be resolved."
        return {"role":"assistant","content":text}

llm = SyntheticLLM()
pprint(llm.complete([{"role":"system","content":SYSTEM},{"role":"user","content":QUESTION}], TOOLS))

{'content': None,
 'role': 'assistant',
 'tool_calls': [{'function': {'arguments': '{"document_id": "INITIAL"}',
                              'name': 'retrieve'},
                 'id': 'call_535e918838',
                 'type': 'function'}]}


### 2 · Application-controlled workflows
Here the application chooses the retrieval and calculation steps **before** asking the model to interpret results. `follow_annex=False` deliberately implements an inadequate total-cash pipeline. Turn it on to repair the application logic. A fixed workflow can follow conditional branches too.

In [3]:
def fixed_workflow(model, store, calculator=False, follow_annex=False):
    messages=[{"role":"system","content":SYSTEM},{"role":"user","content":QUESTION}]
    trace=[]
    packet=retrieve(store,"INITIAL")
    trace.append({"actor":"application","tool":"retrieve","arguments":{"document_id":"INITIAL"},"result":packet})
    messages.append({"role":"user","content":json.dumps(packet)})
    docs={d["id"]:d for d in packet["documents"]}
    requires="Annex A" in docs["COVENANT"]["text"]
    if calculator:
        f=docs["FILING"]["text"]; cash=number(f,"cash")
        if requires and follow_annex:
            annex=retrieve(store,"ANNEX_A")
            trace.append({"actor":"application","tool":"retrieve","arguments":{"document_id":"ANNEX_A"},"result":annex})
            messages.append({"role":"user","content":json.dumps(annex)})
            if not annex["documents"]:
                return {"answer":"HOLD: application cannot establish eligible cash.","trace":trace,"messages":messages}
            cash-=number(annex["documents"][0]["text"],"Restricted cash")
        args={"debt":number(f,"debt"),"eligible_cash":cash,"ebitda":number(f,"EBITDA")}
        result=calculate_leverage(**args)
        trace.append({"actor":"application","tool":"calculate_leverage","arguments":args,"result":result})
        messages.append({"role":"user","content":json.dumps(result)})
    response=model.complete(messages) # No tools offered; application already chose the steps.
    messages.append(response)
    return {"answer":response.get("content"),"trace":trace,"messages":messages}


### 3 · Model-directed workflow
The model requests a call; Python validates and executes it, records the result, and asks the model what to do next. This loop is the same for the synthetic and live clients. Tool access is allowlisted, arguments are checked, and the loop has a budget. No `eval`, arbitrary code execution or trading tool is provided.

In [4]:
def execute_tool(store, name, args):
    if name=="retrieve":
        if set(args)!={"document_id"} or args["document_id"] not in {"INITIAL","FILING","COVENANT","ANNEX_A"}:
            raise ValueError("Invalid retrieval arguments")
        return retrieve(store,**args)
    if name=="calculate_leverage" and set(args)=={"debt","eligible_cash","ebitda"}:
        return calculate_leverage(**args)
    raise ValueError("Tool or arguments not allowed")

def agent_workflow(model, store, max_turns=6):
    messages=[{"role":"system","content":SYSTEM},{"role":"user","content":QUESTION}]
    trace=[]
    for turn in range(max_turns):
        response=model.complete(messages, tools=TOOLS)
        messages.append(response)
        calls=response.get("tool_calls",[])
        if not calls:
            return {"answer":response.get("content"),"trace":trace,"messages":messages}
        if len(calls)>4: raise RuntimeError("Too many calls in one turn")
        for call in calls:
            name=call["function"]["name"]
            args=json.loads(call["function"]["arguments"])
            try: result=execute_tool(store,name,args)
            except (ValueError,TypeError) as exc: result={"error":str(exc)}
            trace.append({"actor":"model request / application execution","tool":name,"arguments":args,"result":result})
            messages.append({"role":"tool","tool_call_id":call["id"],"content":json.dumps(result)})
    return {"answer":"HOLD: tool-loop budget exhausted", "trace":trace,"messages":messages}

store=make_store()
runs={
    "Fixed RAG":fixed_workflow(llm,store),
    "Fixed RAG + calculator":fixed_workflow(llm,store,calculator=True),
    "Agent tools":agent_workflow(llm,store)}
for name,run in runs.items():
    print("\n",name, "\n",run["answer"])
    for step in run["trace"]: print("  ",step["actor"],"→",step["tool"],step["arguments"])



 Fixed RAG 
 Retrieved the filing and covenant, but this workflow supplies no executed calculation. Current headroom needs verification; any referenced annex must be resolved.
   application → retrieve {'document_id': 'INITIAL'}

 Fixed RAG + calculator 
 Calculator reports 3.00× using cash 15.0m. Sources: FILING, COVENANT. Comfort and release still need scenario review and approval.
   application → retrieve {'document_id': 'INITIAL'}
   application → calculate_leverage {'debt': 75.0, 'eligible_cash': 15.0, 'ebitda': 20.0}

 Agent tools 
 Calculator reports 3.25× using cash 10.0m. Sources: FILING, COVENANT, ANNEX_A. Comfort and release still need scenario review and approval.
   model request / application execution → retrieve {'document_id': 'INITIAL'}
   model request / application execution → retrieve {'document_id': 'ANNEX_A'}
   model request / application execution → calculate_leverage {'debt': 75.0, 'eligible_cash': 10.0, 'ebitda': 20.0}


### 4 · Inspect evidence support, independently of the wording
This bounded checker examines executed steps and documents. It does not trust the model's assurance that it checked something. It verifies the current calculation path only—not arbitrary answer truth, completeness of the original source collection, or permission to release.

Notice the difference between **3.00× with correct arithmetic** and **3.25× with supported eligible cash**.

In [5]:
def inspect_path(run):
    docs={d["id"]:d for s in run["trace"] for d in s["result"].get("documents",[])}
    calcs=[s["result"] for s in run["trace"] if "leverage" in s["result"]]
    if "FILING" not in docs or "COVENANT" not in docs: return "HOLD: missing initial sources"
    requires="Annex A" in docs["COVENANT"]["text"]
    if requires and "ANNEX_A" not in docs: return "HOLD: required annex not retrieved"
    if not calcs: return "HOLD: no executed calculation"
    f=docs["FILING"]["text"]
    cash=number(f,"cash")-(number(docs["ANNEX_A"]["text"],"Restricted cash") if requires else 0)
    expected=calculate_leverage(number(f,"debt"),cash,number(f,"EBITDA"))
    if any(not math.isclose(calcs[-1][k],expected[k]) for k in ["debt","eligible_cash","ebitda","leverage"]):
        return "HOLD: calculator inputs/result do not match evidence"
    return "CURRENT CALCULATION SUPPORTED; comfort/release unresolved"

for name,run in runs.items(): print(name, "→",inspect_path(run))
# The actual message transcript: inspect the hand-off, not hidden reasoning.
pprint(runs["Agent tools"]["messages"])


Fixed RAG → HOLD: required annex not retrieved
Fixed RAG + calculator → HOLD: required annex not retrieved
Agent tools → CURRENT CALCULATION SUPPORTED; comfort/release unresolved
[{'content': 'You assess the fictional Harbor Metals case. Use only supplied '
             'documents and tool results.\n'
             'For an agent run, retrieve the initial packet first. Follow any '
             'annex reference before choosing eligible cash.\n'
             'Request calculate_leverage for arithmetic. If evidence is '
             'unavailable, state the gap; do not invent an amount.\n'
             'A current ratio alone does not establish comfortable headroom or '
             'authorization to act.\n'
             'Give a concise answer with document IDs. Tools cannot trade or '
             'release a brief.',
  'role': 'system'},
 {'content': 'Does Harbor Metals have comfortable covenant headroom? Establish '
             'current covenant leverage, cite sources, and identify what '


### 5 · Change the workflow; test the consequences
Before running: predict which change fixes input justification and which leaves a gap.
1. Give the fixed pipeline a programmed annex branch.
2. Remove the annex from the store.
3. Make the synthetic model skip the annex.
4. Change debt to $85m: the executed calculation must change, not replay a saved answer.

Then edit the code yourself: add an additional permitted document, change the tool budget, or make the synthetic model request an unknown tool. Inspect how the controller handles it.

In [6]:
repaired=fixed_workflow(llm,store,calculator=True,follow_annex=True)
missing=agent_workflow(llm,make_store(annex_available=False))
skipped=agent_workflow(SyntheticLLM(skip_annex=True),store)
changed=agent_workflow(llm,make_store(debt=85))
for name,run in [("Programmed branch",repaired),("Missing annex",missing),("Agent skips annex",skipped),("Debt changed",changed)]:
    print(name,"→",inspect_path(run),"\n ",run["answer"])
assert repaired["trace"][-1]["result"]["leverage"]==3.25
assert inspect_path(missing).startswith("HOLD")
assert inspect_path(skipped).startswith("HOLD")
assert changed["trace"][-1]["result"]["leverage"]==3.75
assert calculate_leverage(75,10,20)["leverage"]==3.25
print("Workflow intervention checks passed.")

Programmed branch → CURRENT CALCULATION SUPPORTED; comfort/release unresolved 
  Calculator reports 3.25× using cash 10.0m. Sources: FILING, COVENANT, ANNEX_A. Comfort and release still need scenario review and approval.
Missing annex → HOLD: required annex not retrieved 
  HOLD calculation claim: Annex A is unavailable; eligible cash is unresolved [COVENANT].
Agent skips annex → HOLD: required annex not retrieved 
  Calculator reports 3.00× using cash 15.0m. Sources: FILING, COVENANT. Comfort and release still need scenario review and approval.
Debt changed → CURRENT CALCULATION SUPPORTED; comfort/release unresolved 
  Calculator reports 3.75× using cash 10.0m. Sources: FILING, COVENANT, ANNEX_A. Comfort and release still need scenario review and approval.
Workflow intervention checks passed.


### 6 · Optional: use your own OpenAI or Together key
Keep `PROVIDER="synthetic"` to avoid all network calls. To opt in:
1. Choose `openai` or `together` and a tool-capable model ID available in your account.
2. Set `RUN_LIVE=True` and run the cell; enter the **selected provider's** key through the hidden prompt.
3. Run the next cell to compare the same three workflows.

The Python kernel connects directly to that provider; it does not proxy through BeliefLens. On remote Jupyter, that means the remote kernel sends the request. The question, retrieved fictional documents and tool results leave the kernel. Keys are retained in memory only and never included in the trace; restart the kernel to clear them. Do not type keys into source cells. Live calls incur provider charges and may produce different or incomplete tool paths. There is no automatic synthetic fallback on errors.

The common Chat Completions interface is used here for portability, not a claim of support for every model. See [OpenAI function calling](https://developers.openai.com/api/docs/guides/function-calling) and [Together function calling](https://docs.together.ai/docs/function-calling).

In [7]:
import urllib.request, urllib.error
from getpass import getpass

class HostedLLM:
    ENDPOINTS={"openai":"https://api.openai.com/v1/chat/completions",
               "together":"https://api.together.ai/v1/chat/completions"}
    def __init__(self, provider, model, api_key):
        self.endpoint=self.ENDPOINTS[provider]; self.provider=provider
        self.model=model; self._key=api_key
    def complete(self,messages,tools=None):
        body={"model":self.model,"messages":messages}
        body["max_completion_tokens" if self.provider=="openai" else "max_tokens"]=2048
        if tools: body.update(tools=tools,tool_choice="auto")
        request=urllib.request.Request(self.endpoint,data=json.dumps(body).encode(),headers={
            "Authorization":"Bearer "+self._key,"Content-Type":"application/json","User-Agent":"BeliefLens-Workshop/1.0"})
        try:
            with urllib.request.urlopen(request,timeout=90) as response: result=json.load(response)
        except urllib.error.HTTPError as exc:
            raise RuntimeError(f"Provider HTTP {exc.code}. Check key, model access, tool support and quota. No synthetic fallback.") from None
        except urllib.error.URLError:
            raise RuntimeError("Provider network request failed. No synthetic fallback.") from None
        choice=result["choices"][0]
        if choice.get("finish_reason")=="length": raise RuntimeError("Model hit its token budget; increase it explicitly or choose another model.")
        message=choice["message"]
        # Keep the provider tool-call IDs for subsequent tool messages.
        return {k:message[k] for k in ("role","content","tool_calls") if k in message}

PROVIDER="synthetic" # "openai" or "together"
MODEL=""             # Enter a tool-capable model ID from your provider account.
RUN_LIVE=False
client=SyntheticLLM()
if RUN_LIVE:
    if PROVIDER not in HostedLLM.ENDPOINTS or not MODEL.strip():
        raise ValueError("Select a live provider and model first")
    client=HostedLLM(PROVIDER,MODEL,getpass(f"{PROVIDER} API key (hidden): "))
print("Client selected:",type(client).__name__)

Client selected: SyntheticLLM


In [8]:
if RUN_LIVE:
    live_runs={
        "Fixed RAG":fixed_workflow(client,store),
        "Fixed RAG + calculator":fixed_workflow(client,store,calculator=True),
        "Agent tools":agent_workflow(client,store)}
    for name,run in live_runs.items():
        print("\n",name,"\n",run["answer"],"\n",inspect_path(run))
        pprint(run["trace"])
else:
    print("Live calls disabled. All earlier cells execute real Python workflows with the synthetic model client.")

Live calls disabled. All earlier cells execute real Python workflows with the synthetic model client.


### Learning check
**A.** The calculator correctly returns 3.00×. What still needs checking?
- Its input's definition and evidence, as well as arithmetic. Total cash and covenant-eligible cash differ here.

**B.** Does an agent have exclusive ability to follow an annex?
- No. The repaired fixed pipeline does it too. The difference is who requests the steps: application logic or the model inside a bounded controller.

**C.** What changes when you switch clients?
- Model responses and requested calls can change. Tools, execution limits, trace capture and evidence checks stay in application code.

### Objectives met
- **Architectures:** you wrote and executed both fixed flows and the agent loop.
- **Observability:** you inspected actual messages, arguments and executed tool results.
- **Evidence:** you removed a dependency and checked a deliberately skipped step.
- **Portability:** you can replace the synthetic client with a live provider without replacing orchestration.

**Takeaway:** tool access and plausible language do not establish a supported answer. Inspect which sources were retrieved, how inputs were bound, and which steps actually ran. The scripted model demonstrates those interfaces; live runs test whether a real model uses them.